## Vector DB Data Ingestion Pipeline 

In [ ]:
import os
import sys
from pathlib import Path

# Get project root (parent of notebook folder)
PROJECT_ROOT = Path(os.getcwd()).parent if 'notebook' in os.getcwd() else Path(os.getcwd())
print(f"Project root: {PROJECT_ROOT}")

# Add project root to path for imports
sys.path.insert(0, str(PROJECT_ROOT))

# Load environment variables from project root
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')

# Configuration from .env
PDF_DIR = PROJECT_ROOT / "data" / "pdf_files"
VECTOR_DB_DIR = PROJECT_ROOT / os.getenv("CHROMA_PERSIST_DIR", "data/vector_db")
COLLECTION_NAME = os.getenv("COLLECTION_NAME", "abc_company_docs")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "all-MiniLM-L6-v2")

print(f"PDF Directory: {PDF_DIR}")
print(f"Vector DB Directory: {VECTOR_DB_DIR}")
print(f"Collection Name: {COLLECTION_NAME}")
print(f"Embedding Model: {EMBEDDING_MODEL}")

In [ ]:
from langchain.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [ ]:
## read all the pdfs in the directory

def process_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all the pdfs in pdf_files folder (not recursive - just the folder)
    pdf_files = list(pdf_dir.glob("*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process in {pdf_dir}")
    
    for pdf_file in pdf_files:
        print(f"   - {pdf_file.name}")
    print()

    for pdf_file in pdf_files:
        print(f"Processing: {pdf_file.name}...", end=" ")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['source'] = pdf_file.name  # Also add 'source' for consistency
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"✅ Loaded {len(documents)} pages")

        except Exception as e:
            print(f"❌ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs using the configured path
all_pdf_documents = process_pdfs(PDF_DIR)

In [ ]:
# Show sample of loaded documents
if all_pdf_documents:
    print(f"Sample document:")
    print(f"  Source: {all_pdf_documents[0].metadata.get('source_file', 'unknown')}")
    print(f"  Content preview: {all_pdf_documents[0].page_content[:200]}...")

In [ ]:
## split the texts into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [ ]:
chunks = split_documents(all_pdf_documents)
print(f"\nTotal chunks: {len(chunks)}")

### Embedding and Vector DB

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = None):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name or EMBEDDING_MODEL
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            # Use project models folder for caching
            model_cache = PROJECT_ROOT / "models"
            model_cache.mkdir(exist_ok=True)
            self.model = SentenceTransformer(self.model_name, cache_folder=str(model_cache))
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager
embedding_manager = EmbeddingManager()

### VectorStore

In [ ]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = None, persist_directory: str = None):
        """
        This function initializes the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        # Use configured values from .env
        self.collection_name = collection_name or COLLECTION_NAME
        self.persist_directory = persist_directory or str(VECTOR_DB_DIR)
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persist directory
            os.makedirs(self.persist_directory, exist_ok=True)
            
            # Create client
            self.client = chromadb.PersistentClient(
                path=self.persist_directory,
                settings=Settings(anonymized_telemetry=False)
            )
            
            # Get or create collection with cosine similarity
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"hnsw:space": "cosine", "description": "PDF document embeddings for RAG"}
            )
            
            print(f"Vector store initialized:")
            print(f"  Collection: {self.collection_name}")
            print(f"  Location: {self.persist_directory}")
            print(f"  Existing documents: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def clear_collection(self):
        """Delete and recreate the collection (fresh start)"""
        try:
            self.client.delete_collection(self.collection_name)
            print(f"Deleted existing collection: {self.collection_name}")
        except Exception:
            pass
        
        self.collection = self.client.create_collection(
            name=self.collection_name,
            metadata={"hnsw:space": "cosine", "description": "PDF document embeddings for RAG"}
        )
        print(f"Created fresh collection: {self.collection_name}")

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID for each record
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            # Ensure 'source' key exists for consistency with rag_retriever.py
            if 'source_file' in metadata and 'source' not in metadata:
                metadata['source'] = metadata['source_file']
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection in batches
        batch_size = 100
        for i in range(0, len(ids), batch_size):
            end = min(i + batch_size, len(ids))
            self.collection.add(
                ids=ids[i:end],
                embeddings=embeddings_list[i:end],
                metadatas=metadatas[i:end],
                documents=documents_text[i:end]
            )
            print(f"  Added batch {i//batch_size + 1}/{(len(ids)-1)//batch_size + 1}")
        
        print(f"Successfully added {len(documents)} documents to vector store")
        print(f"Total documents in collection: {self.collection.count()}")

In [ ]:
# Initialize vector store
vectorstore = VectorStore()

# IMPORTANT: Clear existing data and start fresh with all 6 PDFs
print("\n⚠️  Clearing existing collection to reindex all PDFs...")
vectorstore.clear_collection()

In [ ]:
# Extract all text from chunks and create embeddings
texts = [doc.page_content for doc in chunks]

# Generate the Embeddings
embeddings = embedding_manager.generate_embeddings(texts)

# Store in the vector database
vectorstore.add_documents(chunks, embeddings)

### Retrieval From Vector DB

In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [ ]:
# Test retrieval with a query about the NEW PDFs
print("=" * 60)
print("Testing retrieval with new PDF content...")
print("=" * 60)

results = rag_retriever.retrieve("What is the Release Team's monthly budget?")

for r in results:
    print(f"\n[{r['metadata'].get('source', 'unknown')}] (Score: {r['similarity_score']:.3f})")
    print(f"  {r['content'][:200]}...")

In [ ]:
# Test another query
results = rag_retriever.retrieve("What are the budget escalation thresholds?")

for r in results:
    print(f"\n[{r['metadata'].get('source', 'unknown')}] (Score: {r['similarity_score']:.3f})")
    print(f"  {r['content'][:200]}...")

### VectorDB To LLM Output Generation

In [ ]:
# Verify GROQ API key is loaded
groq_key = os.getenv("GROQ_API_KEY")
if groq_key:
    print(f"✅ GROQ_API_KEY loaded (starts with: {groq_key[:10]}...)")
else:
    print("❌ GROQ_API_KEY not found! Check your .env file.")

In [ ]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq

### Initialize the Groq LLM
llm = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name=os.getenv("LLM_MODEL", "llama-3.3-70b-versatile"),
    temperature=0.1,
    max_tokens=1024
)

## Simple RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    # Retrieve the context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    
    if not context:
        return "No relevant context found to answer the question."
    
    # Generate the answer using GROQ LLM
    prompt = f"""Use the following context to answer the question concisely.
Context:
{context}

Question: {query}

Answer:"""
    
    response = llm.invoke([prompt])
    return response.content

In [ ]:
# Test with a question about the NEW PDFs (team budget)
print("=" * 60)
print("Testing RAG with new PDF content...")
print("=" * 60)

answer = rag_simple("What is the Release Team's monthly budget and who is their lead?", rag_retriever, llm)
print(answer)

In [ ]:
# Another test - cost governance
answer = rag_simple("What happens when a team reaches 110% of their budget?", rag_retriever, llm)
print(answer)

### ✅ Setup Complete!

The vector database is now set up with all PDFs from `data/pdf_files/`.

You can now run the main demo:
```bash
uv run python main.py
```